# 🌱 Smart Agriculture Assistant — Vector RAG

A question-answering assistant for smallholder farmers in Rwanda. It answers **only from trusted farming documents** (FAO, IITA, CGIAR, Plantwise, KALRO and others) and cites its sources.

**Pipeline**

```
PDFs → extract text → chunk → embed (bge-small) → FAISS index
                                                     ↓
Farmer question → embed → retrieve top-k chunks → prompt LLM (Qwen3-4B) → answer + sources
```

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then run the cells from top to bottom.

| Section | What it does |
|---|---|
| 0 | Install libraries and settings |
| 1 | Download the documents |
| 2 | Extract and clean text |
| 3 | Chunking |
| 4 | Embeddings and FAISS index |
| 5 | Retrieval |
| 6 | Load the LLM |
| 7 | Generation (the "G" in RAG) |
| 8 | Evaluation |
| 9 | Gradio demo app |
| 10 | Bonus: hybrid search (vector + BM25) |

## 0. Setup

In [1]:
!pip install -q -U pypdf sentence-transformers faiss-cpu gradio "transformers>=4.51" accelerate rank_bm25 groq

In [2]:
import os, re, pickle
import requests
import numpy as np
import torch

DATA_DIR = "/content/agri_docs"     # downloaded PDFs
INDEX_DIR = "/content/agri_index"   # saved FAISS index + chunks
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(INDEX_DIR, exist_ok=True)

# Set to True to use the Groq API instead of Qwen (backup if no GPU is available).
# Needs a free key from console.groq.com saved in Colab Secrets (🔑 icon) as GROQ_API_KEY.
USE_GROQ = False

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


## 1. The knowledge base

Every document here is a publicly available PDF from an agricultural research or extension organisation. Each one is tagged with the crop it covers, so we can cite sources and evaluate retrieval later.

To add more documents, add a line to `DOCUMENTS`, or use the upload cell below.

In [3]:
DOCUMENTS = [
    # ---- Maize (including fall armyworm) ----
    {"file": "maize_faw_ffs_experiences_fao.pdf", "crop": "maize",
     "source": "FAO (2021) Fall armyworm management: Farmer field school experiences in Africa",
     "url": "https://www.fao.org/3/cb3890en/cb3890en.pdf"},
    {"file": "maize_faw_ipm_guidelines_fao.pdf", "crop": "maize",
     "source": "FAO (2021) Guidelines for an IPM strategy for fall armyworm control",
     "url": "https://openknowledge.fao.org/server/api/core/bitstreams/a721f76e-2cf5-4dad-b3b1-6d1c07e3d481/content"},
    {"file": "maize_sustainable_production_manual.pdf", "crop": "maize",
     "source": "CGIAR Manual for Sustainable Maize Production (Ghana)",
     "url": "https://cgspace.cgiar.org/server/api/core/bitstreams/40fbec24-8696-4f7d-b2d1-f6ade042ac7a/content"},
    {"file": "maize_value_chain_rwanda_unido.pdf", "crop": "maize",
     "source": "UNIDO (2013) Rwanda maize value chain analysis",
     "url": "https://downloads.unido.org/ot/53/28/5328232/HABIMANA_EFR_FLD_AFR_RWA_RWA_2013.pdf"},

    # ---- Beans ----
    {"file": "beans_training_manual_maaif_uganda.pdf", "crop": "beans",
     "source": "MAAIF Uganda: Beans training manual for extension workers",
     "url": "https://www.agriculture.go.ug/wp-content/uploads/2020/07/MAAIF-Beans-training-manual-for-extension-workers.pdf"},
    {"file": "beans_production_manual_farm_africa.pdf", "crop": "beans",
     "source": "Farm Africa: Beans production training manual",
     "url": "https://www.farmafrica.org/wp-content/uploads/2025/11/Farm-Africa-Beans-Production-Training-Manual.pdf"},
    {"file": "beans_root_rot_pocket_guide.pdf", "crop": "beans",
     "source": "Research Into Use pocket guide: Bean root rot",
     "url": "https://assets.publishing.service.gov.uk/media/57a08bf4e5274a27b2000ea1/RIU_pocketguide_10.pdf"},

    # ---- Cassava ----
    {"file": "cassava_reference_manual_iita.pdf", "crop": "cassava",
     "source": "IITA (1990) Cassava in Tropical Africa: A Reference Manual",
     "url": "https://cassavabase.org/ftp/ftp/documents/trait_descriptors/IITA_reference_manual_1990.pdf"},

    # ---- Irish potato ----
    {"file": "potato_late_blight_plantwise_kenya.pdf", "crop": "potato",
     "source": "Plantwise factsheet: Potato late blight (Kenya)",
     "url": "https://factsheetadmin.plantwise.org/Uploads/PDFs/20117800552.pdf"},
    {"file": "potato_late_blight_lucid.pdf", "crop": "potato",
     "source": "Pacific Pests, Pathogens and Weeds: Potato late blight",
     "url": "https://apps.lucidcentral.org/ppp_v9/pdf/web_mini/potato_late_blight_265.pdf"},
    {"file": "potato_late_blight_new_brunswick.pdf", "crop": "potato",
     "source": "Government of New Brunswick: Late blight of potato",
     "url": "https://www2.gnb.ca/content/dam/gnb/Departments/10/pdf/Agriculture/potatoes-pommes-de-terre/late-blight-of-potato.pdf"},
    {"file": "potato_sector_policy_rwanda_bpr.pdf", "crop": "potato",
     "source": "BPR Rwanda: Irish potato sector policy",
     "url": "https://findevgateway.org/sites/default/files/publications/files/bpr_irish_potato_sector_policy_-_final.pdf"},
    {"file": "potato_gap_burera_rwanda.pdf", "crop": "potato",
     "source": "Mugisha et al. (2024) Good agricultural practices and Irish potato production, Burera, Rwanda",
     "url": "https://afjare.org/wp-content/uploads/2025/05/2.-Mugisha-et-al.pdf"},

    # ---- Banana ----
    {"file": "banana_bxw_plantwise.pdf", "crop": "banana",
     "source": "Plantwise factsheet: How to recognise banana Xanthomonas wilt",
     "url": "https://factsheetadmin.plantwise.org/Uploads/PDFs/20167800917.pdf"},
    {"file": "banana_bxw_lucid.pdf", "crop": "banana",
     "source": "Pacific Pests, Pathogens, Weeds and Pesticides: Banana xanthomonas wilt",
     "url": "https://apps.lucidcentral.org/pppw_v11/pdf/web_full/banana_xanthomonas_wilt_523.pdf"},
]

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"}

def download(doc):
    path = os.path.join(DATA_DIR, doc["file"])
    if os.path.exists(path) and os.path.getsize(path) > 1000:
        return "already downloaded"
    try:
        r = requests.get(doc["url"], headers=HEADERS, timeout=90, allow_redirects=True)
        if r.status_code != 200:
            return f"FAILED (HTTP {r.status_code})"
        if b"%PDF" not in r.content[:1024]:
            return "FAILED (not a PDF)"
        with open(path, "wb") as f:
            f.write(r.content)
        return f"ok ({len(r.content) / 1e6:.1f} MB)"
    except Exception as e:
        return f"FAILED ({type(e).__name__})"

for doc in DOCUMENTS:
    status = download(doc)
    doc["status"] = status
    print(f"{doc['crop']:8} {doc['file']:45} {status}")

failed = [d["file"] for d in DOCUMENTS if d["status"].startswith("FAILED")]
print("\nPDFs ready:", len([f for f in os.listdir(DATA_DIR) if f.lower().endswith('.pdf')]), "of", len(DOCUMENTS))
if failed:
    print("Some downloads failed (some sites block Colab). Upload agri_docs.zip in the next cell to add them.")

maize    maize_faw_ffs_experiences_fao.pdf             FAILED (HTTP 403)
maize    maize_faw_ipm_guidelines_fao.pdf              FAILED (HTTP 403)
maize    maize_sustainable_production_manual.pdf       ok (4.0 MB)
maize    maize_value_chain_rwanda_unido.pdf            ok (1.8 MB)
beans    beans_training_manual_maaif_uganda.pdf        ok (2.9 MB)
beans    beans_production_manual_farm_africa.pdf       ok (5.5 MB)
beans    beans_root_rot_pocket_guide.pdf               ok (0.2 MB)
cassava  cassava_reference_manual_iita.pdf             ok (5.2 MB)
potato   potato_late_blight_plantwise_kenya.pdf        ok (0.6 MB)
potato   potato_late_blight_lucid.pdf                  ok (0.1 MB)
potato   potato_late_blight_new_brunswick.pdf          ok (0.2 MB)
potato   potato_sector_policy_rwanda_bpr.pdf           ok (0.9 MB)
potato   potato_gap_burera_rwanda.pdf                  ok (0.5 MB)
banana   banana_bxw_plantwise.pdf                      ok (1.4 MB)
banana   banana_bxw_lucid.pdf                     

**If any download failed:** run this cell and upload `agri_docs.zip` (all 15 PDFs in one file). You can also upload single PDFs of your own, such as RAB guides. Skip this cell if everything downloaded.

In [4]:
import zipfile, io
from google.colab import files

uploaded = files.upload()
for name, data in uploaded.items():
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            for member in z.namelist():
                if member.lower().endswith(".pdf"):
                    with open(os.path.join(DATA_DIR, os.path.basename(member)), "wb") as f:
                        f.write(z.read(member))
                    print("Extracted", os.path.basename(member))
    else:
        with open(os.path.join(DATA_DIR, name), "wb") as f:
            f.write(data)
        print("Saved", name)

Saving maize_sustainable_production_manual.pdf to maize_sustainable_production_manual.pdf
Saving maize_value_chain_rwanda_unido.pdf to maize_value_chain_rwanda_unido.pdf
Saving potato_gap_burera_rwanda.pdf to potato_gap_burera_rwanda.pdf
Saving potato_late_blight_lucid.pdf to potato_late_blight_lucid.pdf
Saving potato_late_blight_new_brunswick.pdf to potato_late_blight_new_brunswick.pdf
Saving potato_late_blight_plantwise_kenya.pdf to potato_late_blight_plantwise_kenya.pdf
Saving potato_sector_policy_rwanda_bpr.pdf to potato_sector_policy_rwanda_bpr.pdf
Saving banana_bxw_lucid.pdf to banana_bxw_lucid.pdf
Saving banana_bxw_plantwise.pdf to banana_bxw_plantwise.pdf
Saving beans_production_manual_farm_africa.pdf to beans_production_manual_farm_africa.pdf
Saving beans_root_rot_pocket_guide.pdf to beans_root_rot_pocket_guide.pdf
Saving beans_training_manual_maaif_uganda.pdf to beans_training_manual_maaif_uganda.pdf
Saving cassava_reference_manual_iita.pdf to cassava_reference_manual_iita.pd

## 2. Extract and clean the text

`pypdf` reads each page. We clean up broken lines and extra spaces, and skip pages with almost no text (covers, images, scanned pages).

In [5]:
from pypdf import PdfReader

def clean_text(t):
    t = t.replace("\u00ad", "")                # soft hyphens
    t = re.sub(r"-\n(\w)", r"\1", t)            # re-join words split across lines
    t = re.sub(r"[ \t]+", " ", t)               # collapse spaces and tabs
    t = re.sub(r"\n{2,}", "\n", t)              # collapse blank lines
    return t.strip()

meta = {d["file"]: d for d in DOCUMENTS}
pages = []

for fname in sorted(os.listdir(DATA_DIR)):
    if not fname.lower().endswith(".pdf"):
        continue
    info = meta.get(fname, {"crop": "other", "source": fname})   # uploaded files get "other"
    try:
        reader = PdfReader(os.path.join(DATA_DIR, fname))
    except Exception as e:
        print("Could not read", fname, "->", e)
        continue
    kept = 0
    for i, page in enumerate(reader.pages):
        try:
            text = clean_text(page.extract_text() or "")
        except Exception:
            text = ""
        if len(text.split()) < 30:
            continue
        pages.append({"text": text, "file": fname, "source": info["source"],
                      "crop": info["crop"], "page": i + 1})
        kept += 1
    print(f"{fname:45} {kept:4} / {len(reader.pages):4} pages with text")

print("\nTotal pages with text:", len(pages))

banana_bxw_lucid.pdf                             4 /    4 pages with text
banana_bxw_plantwise.pdf                         1 /    1 pages with text
beans_production_manual_farm_africa.pdf         86 /   89 pages with text
beans_root_rot_pocket_guide.pdf                  7 /    8 pages with text
beans_training_manual_maaif_uganda.pdf          78 /   81 pages with text
cassava_reference_manual_iita.pdf              178 /  184 pages with text
maize_faw_ffs_experiences_fao.pdf               19 /   32 pages with text
maize_faw_ipm_guidelines_fao.pdf                34 /   36 pages with text
maize_sustainable_production_manual.pdf         42 /   45 pages with text
maize_value_chain_rwanda_unido.pdf              46 /   46 pages with text
potato_gap_burera_rwanda.pdf                    14 /   14 pages with text
potato_late_blight_lucid.pdf                     1 /    2 pages with text
potato_late_blight_new_brunswick.pdf             3 /    3 pages with text
potato_late_blight_plantwise_kenya.pdf

## 3. Chunking

LLMs and embedding models work best on short passages, so we split each page into overlapping chunks of about 250 words (~330 tokens) with a 50-word overlap, so a sentence cut at a boundary still appears whole in the next chunk.

Chunking page by page means every chunk keeps an exact page number for citations.

In [6]:
CHUNK_WORDS = 250
OVERLAP_WORDS = 50

def chunk_page(p):
    words = p["text"].split()
    step = CHUNK_WORDS - OVERLAP_WORDS
    out = []
    for start in range(0, len(words), step):
        piece = words[start:start + CHUNK_WORDS]
        if start > 0 and len(piece) < 40:      # skip tiny leftover pieces
            break
        out.append({**p, "text": " ".join(piece)})
        if start + CHUNK_WORDS >= len(words):
            break
    return out

chunks = [c for p in pages for c in chunk_page(p)]
for i, c in enumerate(chunks):
    c["id"] = i

print("Number of chunks:", len(chunks))
print("\nExample chunk:\n", chunks[len(chunks) // 2]["source"], "page", chunks[len(chunks) // 2]["page"])
print(chunks[len(chunks) // 2]["text"][:600], "...")

Number of chunks: 896

Example chunk:
 IITA (1990) Cassava in Tropical Africa: A Reference Manual page 86
or expose them to direct sunlight while the predators are being transported. The predators feed and repro- duce on the leaves for about 5 days. The bags are stored in a cool place. Releases. Once appropriate release fields have been identified, individual plants are selected to receive the released material and are marked with some type of flag. Each phytoseiid species is released in a separate field. Predators are released by placing individual infested leaves, complete with predators and their eggs, on the young, fully developed leaves of the.cassava plant. Taking into account the mortality  ...


## 4. Embeddings and the FAISS index

Each chunk becomes a 384-number vector with `BAAI/bge-small-en-v1.5`. Vectors are normalised, so the inner product equals **cosine similarity**. FAISS stores them and finds the nearest ones quickly.

In [7]:
from sentence_transformers import SentenceTransformer
import faiss

device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)

embeddings = embedder.encode([c["text"] for c in chunks], batch_size=64,
                             show_progress_bar=True, normalize_embeddings=True)
embeddings = np.asarray(embeddings, dtype="float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

faiss.write_index(index, os.path.join(INDEX_DIR, "agri.faiss"))
with open(os.path.join(INDEX_DIR, "chunks.pkl"), "wb") as f:
    pickle.dump(chunks, f)

print("Embedding shape:", embeddings.shape)
print("Vectors in index:", index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Embedding shape: (896, 384)
Vectors in index: 896


## 5. Retrieval

The question is embedded the same way (bge recommends a short instruction prefix for queries), and FAISS returns the `k` most similar chunks.

In [8]:
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def retrieve(question, k=4):
    q = embedder.encode([QUERY_PREFIX + question], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    return [{**chunks[i], "score": float(s)} for s, i in zip(scores[0], ids[0]) if i != -1]

def show(results, n_chars=300):
    for r in results:
        print(f"[{r['score']:.3f}] {r['source']} (page {r['page']})")
        print("   ", r["text"][:n_chars].replace("\n", " "), "...\n")

show(retrieve("How do I stop worms eating my maize leaves?"))

[0.763] CGIAR Manual for Sustainable Maize Production (Ghana) (page 35)
    MANUAL FOR SUSTAINABLE MAIZE PRODUCTION IN GHANA 27 2.3.2 Worms (Cut and Army) and stem borers Management: Farmers should monitor and inspect maize fields immediately after germination to guard against damage caused by cutworms, fall armyworms and stem borers. They should monitor damage on 10 consec ...

[0.725] CGIAR Manual for Sustainable Maize Production (Ghana) (page 36)
    MANUAL FOR SUSTAINABLE MAIZE PRODUCTION IN GHANA 28 2.4 Storage pests of maize and their management 2.4.1 Weevil 2.4.2 Moth 2.4.3 Rodents (Mouse and rat) Damage: They chew into the kernels and lay eggs to cause holes in the kernel to make it hollow and reduce weight of grains. Management: Clean stor ...

[0.714] FAO (2021) Guidelines for an IPM strategy for fall armyworm control (page 18)
    Appendix 1: Integrated pest management strategies for fall armyworm control BACKGROUND Maize (Zea mays) is one of the most important cereals globa

In [9]:
# Try your own questions here
show(retrieve("What does banana wilt look like?"))

[0.733] Plantwise factsheet: How to recognise banana Xanthomonas wilt (page 1)
    FACTSHEETS FOR FARMERS www.plantwise.org Created in Ghana , May 2016 How to Recognise Banana Xanthomonas Wilt Recognize the problem Banana Xanthomonas wilt (BXW) is a devastating bacterial disease of banana and plantain which is widespread in Central and East Africa. Once an outbreak occurs on a far ...

[0.730] Pacific Pests, Pathogens, Weeds and Pesticides: Banana xanthomonas wilt (page 2)
    Detection & inspection Look for the shrivelling of the leaves on the male flower bud that turn black, or premature ripening of fruit on a bunch that otherwise remains green. On older plants, look for yellowing and collapsing leaves, and internal discolouration of the vascular tissues. Also, cut the  ...

[0.699] Pacific Pests, Pathogens, Weeds and Pesticides: Banana xanthomonas wilt (page 1)
    Banana xanthomonas wilt (523) Common Name Banana xanthomonas wilt. It is also known as Xanthomonas wilt, banana bacteri

## 6. Load the LLM

**Qwen3-4B-Instruct-2507** in float16 needs about 8 GB of GPU memory, which fits on a free T4. The first load takes a few minutes.

If `USE_GROQ = True` (section 0), this cell is skipped and the Groq API is used instead.

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_NAME = "Qwen/Qwen3-4B-Instruct-2507"

if not USE_GROQ:
    tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
    llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=torch.float16, device_map="auto")
    llm.eval()
    print("Loaded", LLM_NAME)
else:
    print("Using Groq API, skipping local model.")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Loaded Qwen/Qwen3-4B-Instruct-2507


## 7. Generation: putting the RAG together

The retrieved chunks are numbered and placed in the prompt. The system prompt tells the model to answer **only from that context**, to cite passages, and to admit when it doesn't know. This is what keeps the assistant from inventing advice.

In [11]:
SYSTEM_PROMPT = """You are a smart agriculture assistant for smallholder farmers in Rwanda.
Answer the question using ONLY the numbered context passages provided.
Rules:
- If the context does not contain the answer, reply: "I don't have enough information in my documents to answer that."
- Give practical, clear advice in simple English. Use short steps or bullet points when helpful.
- Cite the passages you used, like [1] or [2].
- Never invent pesticide names, doses, dates or numbers that are not in the context.
- The documents come from several countries. If a passage names an office, agency or phone number from another country, do not tell the farmer to contact it. Say "your local agricultural extension officer or the Rwanda Agriculture Board (RAB)" instead."""

def build_context(results):
    return "\n\n".join(
        f"[{i + 1}] (Source: {r['source']}, page {r['page']})\n{r['text']}"
        for i, r in enumerate(results)
    )

def build_messages(question, context):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]

def generate_qwen(question, context, max_new_tokens=450):
    inputs = tokenizer.apply_chat_template(
        build_messages(question, context), add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens,
                           do_sample=False, repetition_penalty=1.05)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

GROQ_MODEL = "llama-3.3-70b-versatile"   # if this name stops working, pick one from console.groq.com/docs/models

def generate_groq(question, context):
    from groq import Groq
    from google.colab import userdata
    client = Groq(api_key=userdata.get("GROQ_API_KEY"))
    resp = client.chat.completions.create(
        model=GROQ_MODEL, temperature=0.2,
        messages=build_messages(question, context),
    )
    return resp.choices[0].message.content.strip()

def ask(question, k=4, verbose=True):
    results = retrieve(question, k)
    context = build_context(results)
    answer = generate_groq(question, context) if USE_GROQ else generate_qwen(question, context)
    if verbose:
        print("QUESTION:", question, "\n")
        print(answer, "\n\nSOURCES:")
        for i, r in enumerate(results):
            print(f"  [{i + 1}] {r['source']}, page {r['page']}  (similarity {r['score']:.2f})")
    return answer, results

_ = ask("How can I control fall armyworm on my maize farm?")

QUESTION: How can I control fall armyworm on my maize farm? 

To control fall armyworm on your maize farm, follow these practical steps based on the provided context:

- **Monitor your fields regularly**: Check 10 randomly selected sites, inspecting 10 consecutive plants after germination. Look for signs like small holes, "windowpane" feeding, ragged leaves, and moist sawdust-like frass near the leaf whorl [2].  
- **Take action if 20% of plants are infested**: If 20% of the plants show larval damage, apply control measures [2].  
- **Use bio-pesticides or chemical pesticides** as needed to manage the infestation [2].  
- **Practice integrated pest management (IPM)**: Combine methods such as monitoring, using natural predators or parasites, and destroying maize residues after harvest (e.g., by burning) to reduce pest populations [2].  
- **Plough the field** to expose cutworms and other pests to natural predators [2].  
- **Consider intercropping**: Plant other crops between maize rows

In [12]:
_ = ask("My banana fruits are ripening too early and yellow liquid comes out when I cut the stem. What is it and what should I do?")

QUESTION: My banana fruits are ripening too early and yellow liquid comes out when I cut the stem. What is it and what should I do? 

Your banana fruits are ripening too early and yellow liquid comes out when you cut the stem — this is a sign of **Banana Xanthomonas Wilt (BXW)**, a serious bacterial disease [3].

### What it means:
- Premature ripening of fruit (while the rest of the bunch stays green) is a key symptom [3].
- Yellow or brownish bacterial ooze from a cut stem confirms the presence of the disease [1][3].

### What you should do:
1. **Do not replant or share tools** with this plant — the bacteria can spread through cut tools or insects like bees and flies [3].
2. **Remove the infected plant immediately** to prevent spreading the disease to other banana or plantain plants.
3. **Dispose of the plant properly** — do not leave it in the field or compost it, as the bacteria can survive in the soil.
4. **Clean all farming tools** thoroughly with disinfectant (like bleach soluti

In [13]:
# Out-of-scope question: the assistant should say it doesn't know (a good thing to show in the demo)
_ = ask("Who won the 2022 football World Cup?")

QUESTION: Who won the 2022 football World Cup? 

I don't have enough information in my documents to answer that. 

SOURCES:
  [1] FAO (2021) Guidelines for an IPM strategy for fall armyworm control, page 3  (similarity 0.51)
  [2] FAO (2021) Fall armyworm management: Farmer field school experiences in Africa, page 4  (similarity 0.51)
  [3] FAO (2021) Guidelines for an IPM strategy for fall armyworm control, page 7  (similarity 0.50)
  [4] UNIDO (2013) Rwanda maize value chain analysis, page 1  (similarity 0.50)


## 8. Evaluation

We test retrieval on 18 questions. Some use the disease name, and the harder ones describe only what the farmer sees. Each question has the crop it belongs to and a few **key phrases** the right passage should contain.

| Metric | Meaning |
|---|---|
| Crop hit @4 | At least one of the top 4 chunks is from a document about the right crop (easy) |
| Key-phrase hit @1 | The **top** chunk is from the right crop and contains a key phrase (strict) |
| Key-phrase hit @4 | One of the top 4 chunks is from the right crop and contains a key phrase |
| MRR | Mean reciprocal rank: 1.0 if the first correct chunk is ranked 1st, 0.5 if 2nd, and so on |

Then we generate a few answers and check by eye whether they stay faithful to the cited sources.

In [14]:
# (question, crop, key phrases that a correct passage should contain)
EVAL_SET = [
    # Direct questions
    ("How can I control fall armyworm on my maize?", "maize", ["fall armyworm"]),
    ("What is the recommended spacing when planting maize?", "maize", ["75 cm", "75cm"]),
    ("When should I apply fertilizer to maize?", "maize", ["fertili"]),
    ("What is push-pull and how does it help against fall armyworm?", "maize", ["push-pull", "push pull"]),
    ("How should I manage root rot in beans?", "beans", ["root rot"]),
    ("How do I control bean fly?", "beans", ["bean fly", "bean stem maggot"]),
    ("What is rhizobium inoculation for beans?", "beans", ["rhizobi", "inocul"]),
    ("How is cassava mosaic disease spread and controlled?", "cassava", ["mosaic"]),
    ("How do I select healthy cassava stem cuttings for planting?", "cassava", ["cutting"]),
    ("How do I recognise late blight on potatoes?", "potato", ["late blight"]),
    ("Which fungicides are used against potato late blight?", "potato", ["fungicide", "metalaxyl", "mancozeb"]),
    ("What are the main challenges of Irish potato production in Rwanda?", "potato", ["rwanda"]),
    ("What are the symptoms of banana Xanthomonas wilt?", "banana", ["ooze", "wilt"]),
    ("Why should farmers remove the male bud of banana plants?", "banana", ["male bud"]),
    # Harder: describe symptoms without naming the problem
    ("Caterpillars are eating holes in the funnel of my young maize plants. What can I do?", "maize", ["armyworm", "stem borer"]),
    ("My potato leaves have dark wet patches after rainy days and the plants are dying fast.", "potato", ["blight"]),
    ("My bean plants are turning yellow and the roots look brown and rotten.", "beans", ["root rot"]),
    ("How do I stop my knife from spreading disease from one banana plant to another?", "banana", ["tool", "knife", "cutlass", "bleach"]),
]

def is_correct(chunk, crop, phrases):
    text = chunk["text"].lower()
    return chunk["crop"] == crop and any(p in text for p in phrases)

def evaluate(retriever, k=4, show_each=True):
    crop_hits = hit1 = hitk = 0
    rr_total = 0.0
    for q, crop, phrases in EVAL_SET:
        res = retriever(q, k)
        crop_ok = any(r["crop"] == crop for r in res)
        ranks = [i + 1 for i, r in enumerate(res) if is_correct(r, crop, phrases)]
        first = ranks[0] if ranks else None
        crop_hits += crop_ok
        hit1 += first == 1
        hitk += first is not None
        rr_total += 1 / first if first else 0
        if show_each:
            mark = "✅" if first == 1 else ("🟡" if first else "❌")
            print(f"{mark} rank {first or '-'}  {q}\n      top: {res[0]['source']} (p.{res[0]['page']})")
    n = len(EVAL_SET)
    scores = {"crop_hit@k": crop_hits / n, "phrase_hit@1": hit1 / n,
              "phrase_hit@k": hitk / n, "MRR": rr_total / n}
    print(f"\nCrop hit @{k}:        {crop_hits}/{n} = {scores['crop_hit@k']:.0%}")
    print(f"Key-phrase hit @1:  {hit1}/{n} = {scores['phrase_hit@1']:.0%}")
    print(f"Key-phrase hit @{k}:  {hitk}/{n} = {scores['phrase_hit@k']:.0%}")
    print(f"MRR:                {scores['MRR']:.2f}")
    return scores

vector_scores = evaluate(retrieve)

✅ rank 1  How can I control fall armyworm on my maize?
      top: CGIAR Manual for Sustainable Maize Production (Ghana) (p.35)
🟡 rank 2  What is the recommended spacing when planting maize?
      top: Farm Africa: Beans production training manual (p.16)
✅ rank 1  When should I apply fertilizer to maize?
      top: CGIAR Manual for Sustainable Maize Production (Ghana) (p.26)
✅ rank 1  What is push-pull and how does it help against fall armyworm?
      top: FAO (2021) Fall armyworm management: Farmer field school experiences in Africa (p.21)
✅ rank 1  How should I manage root rot in beans?
      top: Research Into Use pocket guide: Bean root rot (p.5)
✅ rank 1  How do I control bean fly?
      top: Farm Africa: Beans production training manual (p.31)
✅ rank 1  What is rhizobium inoculation for beans?
      top: MAAIF Uganda: Beans training manual for extension workers (p.17)
✅ rank 1  How is cassava mosaic disease spread and controlled?
      top: IITA (1990) Cassava in Tropical Africa: 

In [15]:
# Faithfulness check: read each answer and compare it with the sources it cites
for q, _crop, _phrases in [EVAL_SET[0], EVAL_SET[12], EVAL_SET[16]]:
    ask(q)
    print("\n" + "=" * 90 + "\n")

QUESTION: How can I control fall armyworm on my maize? 

To control fall armyworm on your maize, follow these practical steps based on the provided context:

- **Monitor your fields regularly**: Check 10 randomly selected plants in 10 different areas of your field after germination. Look for signs like small holes, "windowpane" feeding, ragged leaves, and moist sawdust-like frass near the leaf whorl [1].  
- **Take action if 20% of plants are infested**: If 20% of the plants show damage, start control measures [1].  
- **Use integrated pest management (IPM)**: Combine methods such as:  
  - **Biological control**: Use natural predators and parasites of fall armyworm [1].  
  - **Biopesticides or chemical pesticides** as needed [1].  
  - **Mechanical control**: Hand-pick or crush eggs and caterpillars when possible — this may help when combined with other methods, though its full effectiveness is still being studied [4].  
- **Improve soil health**: Maintain healthy soil with good orga

## 9. Demo app (Gradio)

`share=True` prints a public link you can open on any device during the presentation. The link lasts while this notebook is running.

In [16]:
import gradio as gr

def chat_fn(message, history):
    answer, results = ask(message, verbose=False)
    sources = "\n".join(f"- [{i + 1}] {r['source']}, page {r['page']}" for i, r in enumerate(results))
    return f"{answer}\n\n**Sources**\n{sources}"

demo = gr.ChatInterface(
    fn=chat_fn,
    title="🌱 Smart Agriculture Assistant",
    description="Ask about maize, cassava, Irish potatoes and bananas. Answers come only from trusted farming guides, with sources.",
    examples=[
        "How can I control fall armyworm on my maize?",
        "What are the signs of late blight on potatoes?",
        "How do I stop banana wilt from spreading?",
        "How do I choose good cassava cuttings?",
    ],
)
demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://80a236c3c3232029b3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 10. Bonus: hybrid search (vector + BM25)

Vector search matches **meaning**, BM25 matches **exact words** (useful for product or disease names). We combine both rankings with **Reciprocal Rank Fusion** and compare the hit rate with plain vector search.

In [17]:
from rank_bm25 import BM25Okapi

def tokenize(t):
    return re.findall(r"[a-z0-9]+", t.lower())

bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])

def hybrid_retrieve(question, k=4, pool=20, rrf_k=60):
    q = embedder.encode([QUERY_PREFIX + question], normalize_embeddings=True).astype("float32")
    _, vec_ids = index.search(q, pool)
    bm_scores = bm25.get_scores(tokenize(question))
    bm_ids = np.argsort(bm_scores)[::-1][:pool]

    fused = {}
    for rank, i in enumerate(vec_ids[0]):
        fused[int(i)] = fused.get(int(i), 0) + 1 / (rrf_k + rank)
    for rank, i in enumerate(bm_ids):
        fused[int(i)] = fused.get(int(i), 0) + 1 / (rrf_k + rank)

    best = sorted(fused, key=fused.get, reverse=True)[:k]
    return [{**chunks[i], "score": fused[i]} for i in best]

hybrid_scores = evaluate(hybrid_retrieve)

print("\n            Vector   Hybrid")
for m in vector_scores:
    print(f"{m:12} {vector_scores[m]:6.2f}   {hybrid_scores[m]:6.2f}")

✅ rank 1  How can I control fall armyworm on my maize?
      top: FAO (2021) Guidelines for an IPM strategy for fall armyworm control (p.5)
🟡 rank 4  What is the recommended spacing when planting maize?
      top: Farm Africa: Beans production training manual (p.16)
✅ rank 1  When should I apply fertilizer to maize?
      top: CGIAR Manual for Sustainable Maize Production (Ghana) (p.26)
✅ rank 1  What is push-pull and how does it help against fall armyworm?
      top: FAO (2021) Fall armyworm management: Farmer field school experiences in Africa (p.21)
✅ rank 1  How should I manage root rot in beans?
      top: Research Into Use pocket guide: Bean root rot (p.5)
✅ rank 1  How do I control bean fly?
      top: Farm Africa: Beans production training manual (p.31)
✅ rank 1  What is rhizobium inoculation for beans?
      top: MAAIF Uganda: Beans training manual for extension workers (p.17)
✅ rank 1  How is cassava mosaic disease spread and controlled?
      top: IITA (1990) Cassava in Trop

## Notes for the presentation

- **Why RAG?** The LLM alone can invent farming advice. RAG grounds every answer in trusted documents and shows the sources.
- **Limitations:** answers are only as good as the documents; scanned PDFs without text are skipped; several sources come from other countries (Ghana, Uganda, Kenya), so the prompt redirects farmers to RAB; English only for now.
- **Next steps:** more Rwanda-specific sources (RAB, MINAGRI), Kinyarwanda support with a multilingual embedding model, a reranker, and SMS or WhatsApp access for farmers.

In [19]:
import os, shutil
from google.colab import files

print(os.listdir(INDEX_DIR))   # should list agri.faiss and chunks.pkl
shutil.make_archive("/content/streamlit_index_files", "zip", INDEX_DIR)
files.download("/content/streamlit_index_files.zip")

['chunks.pkl', 'agri.faiss']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>